# 🎓 Tech Challenge Fase 3: Predição de Alfabetização no Brasil

## Objetivo
Desenvolver um modelo supervisionado capaz de prever se um aluno será considerado **alfabetizado** ou **não alfabetizado**, utilizando variáveis educacionais, territoriais e socioeconômicas.

## Arquitetura de Dados (Medallion Architecture)
- **Bronze**: Dados brutos importados das fontes originais
- **Silver**: Dados limpos, padronizados e validados
- **Gold**: Dados agregados, enriquecidos e prontos para análise/modelagem

## Fluxo de Enriquecimento
```
df_ts_aluno.csv (base de alunos)
    |
    +-- merge por id_municipio --> ibge_populacao_municipio.csv
    +-- merge por id_municipio --> ibge_pib_municipio.csv  
    +-- merge por id_municipio --> inep_ideb_municipio.csv
    +-- merge por id_municipio --> censo_escolar_escola.csv (média por município)
    +-- merge por id_municipio --> inep_indicadores_educacionais_escola_inse.csv (média por município)
    |
    v
gold_alunos_enriquecido.csv (base final para EDA e modelagem)
```

---
## 1. Configuração e Imports

In [ ]:
# Bibliotecas principais
import pandas as pd
import numpy as np
import os
import re

# Bibliotecas de visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
pd.options.display.max_rows = 100

# Tema profissional para gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Paleta de cores profissional
CORES = {
    'primaria': '#2E86AB',
    'secundaria': '#A23B72',
    'sucesso': '#28A745',
    'perigo': '#DC3545',
    'alerta': '#FFC107',
    'info': '#17A2B8',
    'escuro': '#343A40',
    'claro': '#F8F9FA'
}

# Caminhos dos dados
DADOS_ANTIGOS = 'dados/antigos/'
DADOS_NOVOS = 'dados/novos/'
DADOS_GOLD = 'dados/gold/'

# Criar pasta gold se não existir
os.makedirs(DADOS_GOLD, exist_ok=True)

print("✅ Bibliotecas carregadas com sucesso!")
print(f"📁 Diretório de trabalho: {os.getcwd()}")

---
## 2. Funções Auxiliares

In [ ]:
# =============================================================================
# DICIONÁRIO DE PADRONIZAÇÃO DE COLUNAS
# =============================================================================

COLUNAS_PADRONIZADAS = {
    'NU_ANO_AVALIACAO': 'ano_avaliacao',
    'CO_UF': 'cod_uf',
    'SG_UF': 'sigla_uf',
    'ID_ALUNO': 'id_aluno',
    'TP_SERIE': 'tipo_serie',
    'ID_ESCOLA': 'id_escola',
    'TP_DEPENDENCIA': 'tipo_dependencia',
    'NO_MUNICIPIO': 'nome_municipio',
    'IN_PRESENCA_LP': 'ind_presenca_lp',
    'IN_PREENCHIMENTO_LP': 'ind_preenchimento_lp',
    'CO_CADERNO_LP': 'cod_caderno_lp',
    'VL_PESO_ALUNO_LP': 'peso_aluno_lp',
    'VL_PROFICIENCIA_LP': 'proficiencia_lp',
    'IN_ALFABETIZADO': 'ind_alfabetizado',
}

print(f"📋 Dicionário de {len(COLUNAS_PADRONIZADAS)} colunas definido.")

In [ ]:
# =============================================================================
# FUNÇÕES DE LIMPEZA E TRANSFORMAÇÃO
# =============================================================================

def padronizar_nomes_colunas(df: pd.DataFrame, mapeamento: dict = None) -> pd.DataFrame:
    df = df.copy()
    if mapeamento:
        df = df.rename(columns={k: v for k, v in mapeamento.items() if k in df.columns})
    novas_colunas = {col: col.lower().strip() for col in df.columns}
    return df.rename(columns=novas_colunas)

def converter_id_para_int64(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    df = df.copy()
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    return df

def resumo_dataframe(df: pd.DataFrame, nome: str = "DataFrame") -> None:
    print(f"\n{'='*60}")
    print(f"📊 RESUMO: {nome}")
    print(f"{'='*60}")
    print(f"Linhas: {len(df):,}")
    print(f"Colunas: {df.shape[1]}")
    print(f"Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

def log_merge(nome_merge: str, antes: int, depois: int, chave: str):
    diff = depois - antes
    print(f"\n🔗 MERGE: {nome_merge}")
    print(f"   Chave: {chave}")
    print(f"   Linhas antes: {antes:,}")
    print(f"   Linhas depois: {depois:,}")

def remover_colunas_anos_antigos(df, ano_manter='2023'):
    """
    Remove colunas que contêm anos diferentes do especificado.
    Padrão: busca sufixo _YYYY no final do nome da coluna.
    """
    colunas_remover = []
    padrao_ano = re.compile(r'_(20\d{2})$')
    
    for col in df.columns:
        match = padrao_ano.search(col)
        if match:
            ano = match.group(1)
            if ano != ano_manter:
                colunas_remover.append(col)
    
    if colunas_remover:
        print(f"\n🗑️ REMOVENDO {len(colunas_remover)} COLUNAS COM ANOS ≠ {ano_manter}:")
        for col in sorted(colunas_remover):
            print(f"   ❌ {col}")
        df = df.drop(columns=colunas_remover)
        
        colunas_mantidas = [c for c in df.columns if ano_manter in c]
        print(f"\n✅ COLUNAS COM {ano_manter} MANTIDAS:")
        for col in colunas_mantidas:
            print(f"   ✓ {col}")
    
    return df

print("✅ Funções auxiliares definidas!")

---
## 3. Camada Bronze - Importação de Dados

In [ ]:
# =============================================================================
# BRONZE: BASE PRINCIPAL DE ALUNOS
# =============================================================================
print("📥 Carregando base de alunos...")
bronze_alunos = pd.read_csv(
    DADOS_ANTIGOS + "TS_ALUNO.csv",
    sep=';',
    encoding='latin-1',
    engine='python',
    on_bad_lines='warn'
)
# Renomear CO_MUNICIPIO para id_municipio se necessário
if 'CO_MUNICIPIO' in bronze_alunos.columns and 'id_municipio' not in bronze_alunos.columns:
    bronze_alunos = bronze_alunos.rename(columns={'CO_MUNICIPIO': 'id_municipio'})
resumo_dataframe(bronze_alunos, "bronze_alunos")
print(f"\n📍 UFs disponíveis: {bronze_alunos['SG_UF'].nunique()} estados")
print(f"   {sorted(bronze_alunos['SG_UF'].unique().tolist())}")

In [ ]:
# =============================================================================
# BRONZE: BASES NOVAS PARA ENRIQUECIMENTO
# =============================================================================

print("📥 Carregando bases de enriquecimento...\n")

bronze_populacao = pd.read_csv(DADOS_NOVOS + "ibge_populacao_municipio.csv")
print(f"✓ População: {len(bronze_populacao):,} registros")

bronze_pib = pd.read_csv(DADOS_NOVOS + "ibge_pib_municipio.csv")
print(f"✓ PIB: {len(bronze_pib):,} registros")

bronze_indicadores = pd.read_csv(DADOS_NOVOS + "inep_indicadores_educacionais_escola_inse.csv")
print(f"✓ Indicadores Educacionais: {len(bronze_indicadores):,} registros")

bronze_censo = pd.read_csv(DADOS_NOVOS + "censo_escolar_escola.csv", low_memory=False)
print(f"✓ Censo Escolar: {len(bronze_censo):,} registros")

bronze_ideb = pd.read_csv(DADOS_NOVOS + "inep_ideb_municipio.csv")
print(f"✓ IDEB: {len(bronze_ideb):,} registros")

print("\n✅ Todas as bases carregadas!")

---
## 4. Camada Silver - Limpeza e Padronização

In [ ]:
# =============================================================================
# SILVER: PADRONIZAÇÃO E LIMPEZA DA BASE DE ALUNOS
# =============================================================================

silver_alunos = bronze_alunos.copy()

# Padronizar nomes de colunas
silver_alunos = padronizar_nomes_colunas(silver_alunos, COLUNAS_PADRONIZADAS)

# Converter IDs para Int64
silver_alunos = converter_id_para_int64(silver_alunos, ['id_municipio', 'id_escola', 'cod_uf'])

# Filtrar dependências válidas
silver_alunos = silver_alunos[silver_alunos['tipo_dependencia'].isin([1, 2, 3, 4])]

# Remover duplicatas
silver_alunos = silver_alunos.drop_duplicates(subset=['ano_avaliacao', 'id_aluno'], keep='first')

# ⚠️ REMOVER COLUNAS COM ANOS DIFERENTES DE 2023
silver_alunos = remover_colunas_anos_antigos(silver_alunos, '2023')

resumo_dataframe(silver_alunos, "silver_alunos (após limpeza)")

In [ ]:
# =============================================================================
# SILVER: PADRONIZAÇÃO DAS BASES DE ENRIQUECIMENTO
# =============================================================================

# INDICADORES EDUCACIONAIS (INSE)
# NOTA: id_escola em ts_aluno é fictício, então agregamos por id_municipio (média)
silver_indicadores = bronze_indicadores.copy()
silver_indicadores = padronizar_nomes_colunas(silver_indicadores)
silver_indicadores = converter_id_para_int64(silver_indicadores, ['id_municipio', 'id_escola', 'ano'])
silver_indicadores = silver_indicadores[silver_indicadores['ano'] == 2023]

# Colunas numéricas para agregar por município
colunas_numericas_inse = [
    'inse', 'quantidade_alunos_inse',
    'percentual_nivel_1', 'percentual_nivel_2', 'percentual_nivel_3', 'percentual_nivel_4',
    'percentual_nivel_5', 'percentual_nivel_6', 'percentual_nivel_7', 'percentual_nivel_8'
]
colunas_disponiveis_inse = [c for c in colunas_numericas_inse if c in silver_indicadores.columns]

# Agregar por município: calcular média dos indicadores de todas as escolas do município
if 'id_municipio' in silver_indicadores.columns and len(colunas_disponiveis_inse) > 0:
    silver_indicadores = silver_indicadores.groupby('id_municipio')[colunas_disponiveis_inse].mean().reset_index()
    # Renomear colunas para prefixo inse_
    colunas_renomear_inse = {c: f'inse_{c}' if not c.startswith('inse') and c != 'id_municipio' else c for c in silver_indicadores.columns}
    silver_indicadores = silver_indicadores.rename(columns=colunas_renomear_inse)
    print(f"✅ Indicadores INSE: {len(silver_indicadores):,} municípios (média por município)")
else:
    silver_indicadores = pd.DataFrame({'id_municipio': pd.array([], dtype='Int64')})
    print("⚠️ Indicadores INSE: Dados insuficientes para agregação")

# CENSO ESCOLAR
# NOTA: id_escola em ts_aluno é fictício, então agregamos por id_municipio (média)
silver_censo = bronze_censo.copy()
silver_censo = padronizar_nomes_colunas(silver_censo)
silver_censo = converter_id_para_int64(silver_censo, ['id_municipio', 'id_escola', 'ano'])
silver_censo = silver_censo[silver_censo['ano'] == 2023]

colunas_censo = [
    'id_municipio', 'agua_filtrada', 'agua_potavel', 'energia_rede_publica',
    'biblioteca', 'laboratorio_informatica', 'quadra_esportes',
    'sala_leitura', 'parque_infantil', 'patio_coberto',
    'internet', 'banda_larga', 'alimentacao'
]
colunas_disponiveis_censo = [c for c in colunas_censo if c in silver_censo.columns]

# Agregar por município: calcular média dos indicadores de todas as escolas do município
colunas_num_censo = [c for c in colunas_disponiveis_censo if c != 'id_municipio']
if 'id_municipio' in silver_censo.columns and len(colunas_num_censo) > 0:
    silver_censo = silver_censo.groupby('id_municipio')[colunas_num_censo].mean().reset_index()
    colunas_renomear_censo = {c: f'censo_{c}' for c in silver_censo.columns if c != 'id_municipio'}
    silver_censo = silver_censo.rename(columns=colunas_renomear_censo)
    print(f"✅ Censo Escolar: {len(silver_censo):,} municípios (média por município)")
else:
    silver_censo = pd.DataFrame({'id_municipio': pd.array([], dtype='Int64')})
    print("⚠️ Censo Escolar: Dados insuficientes para agregação")

# IDEB - Tratamento robusto
silver_ideb = bronze_ideb.copy()
silver_ideb = padronizar_nomes_colunas(silver_ideb)
silver_ideb = converter_id_para_int64(silver_ideb, ['id_municipio', 'ano'])

# Filtrar apenas anos iniciais (sem filtro de ano ou rede)
mask_iniciais = silver_ideb['anos_escolares'].str.contains('iniciais', case=False, na=False)
silver_ideb = silver_ideb[mask_iniciais].copy()

colunas_ideb = ['id_municipio', 'ideb', 'nota_saeb_matematica', 'nota_saeb_lingua_portuguesa', 'taxa_aprovacao']
colunas_disponiveis = [c for c in colunas_ideb if c in silver_ideb.columns]
silver_ideb = silver_ideb[colunas_disponiveis].dropna(subset=['id_municipio'])

colunas_num = [c for c in colunas_disponiveis if c != 'id_municipio']
if len(silver_ideb) > 0 and len(colunas_num) > 0:
    silver_ideb = silver_ideb.groupby('id_municipio')[colunas_num].mean().reset_index()
    colunas_renomear = {c: f'ideb_{c}' for c in silver_ideb.columns if c != 'id_municipio'}
    silver_ideb = silver_ideb.rename(columns=colunas_renomear)
    print(f"✅ IDEB: {len(silver_ideb):,} municípios")
else:
    silver_ideb = pd.DataFrame({'id_municipio': pd.array([], dtype='Int64')})
    print("⚠️ IDEB: Dados insuficientes para merge")

# POPULAÇÃO E PIB
silver_populacao = bronze_populacao.copy()
silver_populacao = padronizar_nomes_colunas(silver_populacao)
silver_populacao = converter_id_para_int64(silver_populacao, ['id_municipio', 'ano'])
silver_populacao = silver_populacao[silver_populacao['ano'] == 2023][['id_municipio', 'populacao']]
silver_populacao = silver_populacao.rename(columns={'populacao': 'populacao_2023'})
print(f"✅ População: {len(silver_populacao):,} municípios")

silver_pib = bronze_pib.copy()
silver_pib = padronizar_nomes_colunas(silver_pib)
silver_pib = converter_id_para_int64(silver_pib, ['id_municipio', 'ano'])
silver_pib = silver_pib[silver_pib['ano'] == 2023][['id_municipio', 'pib']]
silver_pib = silver_pib.rename(columns={'pib': 'pib_2023'})
print(f"✅ PIB: {len(silver_pib):,} municípios")

---
## 5. Camada Gold - Enriquecimento e Merges

In [ ]:
# =============================================================================
# GOLD: MERGES E ENRIQUECIMENTO
# =============================================================================

gold_alunos = silver_alunos.copy()
print(f"📊 Base inicial: {len(gold_alunos):,} alunos x {len(gold_alunos.columns)} colunas")

# Verificar colunas-chave
print(f"\n🔑 Verificando colunas-chave:")
print(f"   id_municipio em gold_alunos: {'id_municipio' in gold_alunos.columns}")

# Merge 1: Indicadores Educacionais (INSE) - por id_municipio
if 'id_municipio' in gold_alunos.columns and 'id_municipio' in silver_indicadores.columns:
    gold_alunos = gold_alunos.merge(silver_indicadores, on='id_municipio', how='left')
    print(f"✓ Após merge Indicadores INSE: {len(gold_alunos.columns)} colunas")
else:
    print("⚠️ Merge Indicadores INSE pulado - coluna id_municipio não encontrada")

# Merge 2: Censo Escolar - por id_municipio
if 'id_municipio' in gold_alunos.columns and 'id_municipio' in silver_censo.columns:
    gold_alunos = gold_alunos.merge(silver_censo, on='id_municipio', how='left')
    print(f"✓ Após merge Censo: {len(gold_alunos.columns)} colunas")
else:
    print("⚠️ Merge Censo pulado - coluna id_municipio não encontrada")

# Merge 3: IDEB
if 'id_municipio' in gold_alunos.columns and 'id_municipio' in silver_ideb.columns:
    gold_alunos = gold_alunos.merge(silver_ideb, on='id_municipio', how='left')
    print(f"✓ Após merge IDEB: {len(gold_alunos.columns)} colunas")
else:
    print(f"⚠️ Merge IDEB pulado - id_municipio em gold: {'id_municipio' in gold_alunos.columns}, em silver_ideb: {'id_municipio' in silver_ideb.columns}")

# Merge 4: População
if 'id_municipio' in gold_alunos.columns and 'id_municipio' in silver_populacao.columns:
    gold_alunos = gold_alunos.merge(silver_populacao, on='id_municipio', how='left')
    print(f"✓ Após merge População: {len(gold_alunos.columns)} colunas")
else:
    print(f"⚠️ Merge População pulado")

# Merge 5: PIB
if 'id_municipio' in gold_alunos.columns and 'id_municipio' in silver_pib.columns:
    gold_alunos = gold_alunos.merge(silver_pib, on='id_municipio', how='left')
    print(f"✓ Após merge PIB: {len(gold_alunos.columns)} colunas")
else:
    print(f"⚠️ Merge PIB pulado")

print(f"\n✅ Base final: {len(gold_alunos):,} alunos x {len(gold_alunos.columns)} colunas")

In [ ]:
# =============================================================================
# GOLD: FEATURE ENGINEERING
# =============================================================================

# Mapeamento UF -> Região
UF_PARA_REGIAO = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 
    'PE': 'Nordeste', 'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

# Criar coluna de região
if 'sigla_uf' in gold_alunos.columns:
    gold_alunos['nome_regiao'] = gold_alunos['sigla_uf'].map(UF_PARA_REGIAO)
    print(f"✅ Coluna nome_regiao criada! Regiões: {gold_alunos['nome_regiao'].unique().tolist()}")

# 1. Label de dependência administrativa
gold_alunos['desc_dependencia'] = gold_alunos['tipo_dependencia'].map({
    1: 'Federal', 2: 'Estadual', 3: 'Municipal', 4: 'Privada'
})

# 2. PIB per capita
if 'populacao_2023' in gold_alunos.columns and 'pib_2023' in gold_alunos.columns:
    gold_alunos['pib_per_capita'] = gold_alunos['pib_2023'] / gold_alunos['populacao_2023'].replace(0, np.nan)

# 3. Score de infraestrutura
colunas_infra = [c for c in gold_alunos.columns if c.startswith('censo_') and c != 'censo_id_escola']
if colunas_infra:
    gold_alunos['score_infraestrutura'] = gold_alunos[colunas_infra].sum(axis=1)
    gold_alunos['score_infraestrutura_pct'] = (gold_alunos['score_infraestrutura'] / len(colunas_infra) * 100).round(1)

print("✅ Features derivadas criadas!")
print(f"   - nome_regiao")
print(f"   - desc_dependencia")
print(f"   - pib_per_capita")
print(f"   - score_infraestrutura")

---
## 6. Exportação da Base Enriquecida

In [ ]:
# =============================================================================
# EXPORTAÇÃO DA BASE GOLD
# =============================================================================

arquivo_saida = DADOS_GOLD + 'gold_alunos_enriquecido.csv'
gold_alunos.to_csv(arquivo_saida, index=False)

tamanho_mb = os.path.getsize(arquivo_saida) / (1024 * 1024)

print(f"\n✅ BASE EXPORTADA COM SUCESSO!")
print(f"   📁 Arquivo: {arquivo_saida}")
print(f"   📊 Tamanho: {tamanho_mb:.2f} MB")
print(f"   📋 Linhas: {len(gold_alunos):,}")
print(f"   📋 Colunas: {len(gold_alunos.columns)}")

---
# 📊 7. Análise Exploratória de Dados (EDA)

Análise completa da base enriquecida com visualizações profissionais para apresentação.

In [ ]:
# =============================================================================
# CONFIGURAÇÃO DO EDA
# =============================================================================

df = gold_alunos.copy()

# Paleta de cores para gráficos
cor_alfabetizado = '#2E86AB'
cor_nao_alfabetizado = '#E94F37'
cor_destaque = '#F39237'
paleta_regioes = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']

print(f"📊 BASE PARA ANÁLISE")
print(f"{'='*50}")
print(f"Total de registros: {len(df):,}")
print(f"Total de colunas: {len(df.columns)}")
print(f"\n📋 Colunas disponíveis:")
print(list(df.columns))

### 7.1 📈 Visão Geral da Variável Alvo

In [ ]:
# =============================================================================
# DISTRIBUIÇÃO DA VARIÁVEL ALVO - GRÁFICO PROFISSIONAL
# =============================================================================
# Calcular estatísticas
total = len(df)
alfabetizados = df['ind_alfabetizado'].sum()
nao_alfabetizados = total - alfabetizados
taxa_alfabetizacao = (alfabetizados / total * 100)

# Criar figura com subplots
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "bar"}, {"type": "pie"}]],
    subplot_titles=('<b>Distribuição Absoluta</b>', '<b>Distribuição Percentual</b>'),
    horizontal_spacing=0.15
)

# Gráfico de barras
fig.add_trace(
    go.Bar(
        x=['Não Alfabetizado', 'Alfabetizado'],
        y=[nao_alfabetizados, alfabetizados],
        marker=dict(
            color=[cor_nao_alfabetizado, cor_alfabetizado],
            line=dict(color='white', width=2)
        ),
        text=[f'<b>{nao_alfabetizados:,}</b><br>({nao_alfabetizados/total*100:.1f}%)', 
              f'<b>{alfabetizados:,}</b><br>({alfabetizados/total*100:.1f}%)'],
        textposition='outside',
        textfont=dict(size=13, color='#333'),
        showlegend=False,
        width=0.6
    ),
    row=1, col=1
)

# Gráfico de rosca (donut)
fig.add_trace(
    go.Pie(
        labels=['Alfabetizado', 'Não Alfabetizado'],
        values=[alfabetizados, nao_alfabetizados],
        marker=dict(
            colors=[cor_alfabetizado, cor_nao_alfabetizado],
            line=dict(color='white', width=3)
        ),
        textinfo='percent+label',
        textfont=dict(size=13, color='white'),
        textposition='inside',
        hole=0.45,
        pull=[0.03, 0.03],
        rotation=90
    ),
    row=1, col=2
)

# Layout melhorado
fig.update_layout(
    title=dict(
        text='<b>🎯 Distribuição da Variável Alvo: Alfabetização</b>',
        font=dict(size=22, color='#1a1a2e'),
        x=0.5,
        y=0.95
    ),
    height=520,
    margin=dict(t=100, b=80, l=60, r=60),
    showlegend=False,
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial, sans-serif')
)

# Ajustar eixo Y para não cortar texto
fig.update_yaxes(
    range=[0, max(nao_alfabetizados, alfabetizados) * 1.25],
    row=1, col=1,
    gridcolor='#eee',
    tickformat=','
)

fig.update_xaxes(
    tickfont=dict(size=12, color='#333'),
    row=1, col=1
)

# Ajustar títulos dos subplots
fig.update_annotations(font=dict(size=14, color='#333'))

fig.show()

# Resumo estatístico
print(f"\n{'='*60}")
print(f"📊 RESUMO DA VARIÁVEL ALVO")
print(f"{'='*60}")
print(f"   Total de alunos: {total:,}")
print(f"   Alfabetizados: {alfabetizados:,} ({taxa_alfabetizacao:.2f}%)")
print(f"   Não Alfabetizados: {nao_alfabetizados:,} ({100-taxa_alfabetizacao:.2f}%)")
print(f"\n   ⚠️ Balanceamento: {'Equilibrado' if 40 < taxa_alfabetizacao < 60 else 'Desbalanceado'}")

### 7.2 🗺️ Análise Geográfica

In [ ]:
# =============================================================================
# TAXA DE ALFABETIZAÇÃO POR UF - ANÁLISE COMPLETA
# =============================================================================
# Calcular taxa por UF
taxa_uf = df.groupby('sigla_uf').agg(
    total_alunos=('ind_alfabetizado', 'count'),
    alfabetizados=('ind_alfabetizado', 'sum')
).reset_index()
taxa_uf['taxa_alfabetizacao'] = (taxa_uf['alfabetizados'] / taxa_uf['total_alunos'] * 100).round(2)
taxa_uf = taxa_uf.sort_values('taxa_alfabetizacao', ascending=False)
media_nacional = taxa_uf['taxa_alfabetizacao'].mean()

n_ufs = len(taxa_uf)
print(f"📍 Total de UFs nos dados: {n_ufs}")

# Definir cores baseadas na média (azul = acima, vermelho = abaixo)
cores_uf = ['#2E86AB' if x >= media_nacional else '#E94F37' for x in taxa_uf['taxa_alfabetizacao']]

fig = go.Figure()

# Barras verticais
fig.add_trace(go.Bar(
    x=taxa_uf['sigla_uf'],
    y=taxa_uf['taxa_alfabetizacao'],
    marker=dict(
        color=cores_uf,
        line=dict(color='white', width=2)
    ),
    text=taxa_uf['taxa_alfabetizacao'].apply(lambda x: f'<b>{x:.1f}%</b>'),
    textposition='outside',
    textfont=dict(size=14, color='#333'),
    hovertemplate='<b>%{x}</b><br>Taxa: %{y:.1f}%<br>Alunos: %{customdata:,}<extra></extra>',
    customdata=taxa_uf['total_alunos'],
    showlegend=False
))

# Linha da média nacional
fig.add_hline(
    y=media_nacional, 
    line_dash="dash", 
    line_color="#F39237",
    line_width=3,
    annotation_text=f"Média: {media_nacional:.1f}%",
    annotation_position="top right",
    annotation_font_size=12,
    annotation_font_color="#F39237"
)

fig.update_layout(
    title=dict(
        text='<b>🗺️ Taxa de Alfabetização por Unidade Federativa</b>',
        font=dict(size=20, color='#1a1a2e'),
        x=0.5,
        y=0.95
    ),
    xaxis_title='Estado',
    yaxis_title='Taxa de Alfabetização (%)',
    height=500,
    margin=dict(t=100, b=80, l=60, r=40),
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(
        gridcolor='#E5E5E5',
        tickfont=dict(size=12)
    ),
    yaxis=dict(
        gridcolor='#E5E5E5',
        range=[0, max(taxa_uf['taxa_alfabetizacao']) * 1.15],
        ticksuffix='%',
        dtick=10
    ),
    bargap=0.3
)

# Adicionar legenda customizada
fig.add_annotation(
    x=0.02, y=0.98, xref='paper', yref='paper',
    text='🔵 Acima da média  🔴 Abaixo da média',
    showarrow=False, font=dict(size=11, color='#666'),
    bgcolor='white', borderpad=4
)

fig.show()

# Resumo
print(f"\n{'='*60}")
print(f"📊 RESUMO - TAXA DE ALFABETIZAÇÃO POR UF")
print(f"{'='*60}")
print(f"   Média das UFs: {media_nacional:.1f}%")
print(f"   Total de UFs analisadas: {n_ufs}")
print(f"\n📈 RANKING COMPLETO:")
for rank, (_, row) in enumerate(taxa_uf.iterrows(), 1):
    status = '🔵' if row['taxa_alfabetizacao'] >= media_nacional else '🔴'
    print(f"   {rank}º {status} {row['sigla_uf']}: {row['taxa_alfabetizacao']:.1f}% ({row['total_alunos']:,} alunos)")

In [ ]:
# =============================================================================
# TAXA DE ALFABETIZAÇÃO POR REGIÃO
# =============================================================================

if 'nome_regiao' in df.columns:
    taxa_regiao = df.groupby('nome_regiao').agg(
        total_alunos=('ind_alfabetizado', 'count'),
        alfabetizados=('ind_alfabetizado', 'sum')
    ).reset_index()
    taxa_regiao['taxa_alfabetizacao'] = (taxa_regiao['alfabetizados'] / taxa_regiao['total_alunos'] * 100).round(2)
    taxa_regiao = taxa_regiao.sort_values('taxa_alfabetizacao', ascending=False)
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=taxa_regiao['nome_regiao'],
        y=taxa_regiao['taxa_alfabetizacao'],
        marker_color=paleta_regioes[:len(taxa_regiao)],
        text=taxa_regiao['taxa_alfabetizacao'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        textfont=dict(size=14, color='black'),
        hovertemplate='<b>%{x}</b><br>Taxa: %{y:.1f}%<br>Alunos: %{customdata:,}<extra></extra>',
        customdata=taxa_regiao['total_alunos']
    ))
    
    fig.update_layout(
        title=dict(
            text='<b>🌎 Taxa de Alfabetização por Região</b>',
            font=dict(size=18),
            x=0.5
        ),
        xaxis_title='',
        yaxis_title='Taxa de Alfabetização (%)',
        height=450,
        paper_bgcolor='white',
        plot_bgcolor='white',
        yaxis=dict(gridcolor='#E5E5E5', range=[0, 100])
    )
    
    fig.show()
else:
    print("⚠️ Coluna 'nome_regiao' não encontrada")

### 7.3 🏫 Análise por Dependência Administrativa

In [ ]:
# =============================================================================
# ANÁLISE POR DEPENDÊNCIA ADMINISTRATIVA
# =============================================================================

taxa_dep = df.groupby('desc_dependencia').agg(
    total_alunos=('ind_alfabetizado', 'count'),
    alfabetizados=('ind_alfabetizado', 'sum')
).reset_index()
taxa_dep['taxa_alfabetizacao'] = (taxa_dep['alfabetizados'] / taxa_dep['total_alunos'] * 100).round(2)
taxa_dep = taxa_dep.sort_values('taxa_alfabetizacao', ascending=False)

# Criar subplots
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "bar"}, {"type": "pie"}]],
    subplot_titles=('<b>Taxa de Alfabetização</b>', '<b>Distribuição de Alunos</b>')
)

cores_dep = ['#2E86AB', '#A23B72', '#F18F01', '#3B1F2B']

# Gráfico de barras
fig.add_trace(
    go.Bar(
        x=taxa_dep['desc_dependencia'],
        y=taxa_dep['taxa_alfabetizacao'],
        marker_color=cores_dep[:len(taxa_dep)],
        text=taxa_dep['taxa_alfabetizacao'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        textfont=dict(size=13),
        showlegend=False
    ),
    row=1, col=1
)

# Gráfico de pizza
fig.add_trace(
    go.Pie(
        labels=taxa_dep['desc_dependencia'],
        values=taxa_dep['total_alunos'],
        marker_colors=cores_dep[:len(taxa_dep)],
        textinfo='percent+label',
        textfont=dict(size=12),
        hole=0.3
    ),
    row=1, col=2
)

fig.update_layout(
    title=dict(
        text='<b>🏫 Análise por Dependência Administrativa</b>',
        font=dict(size=18),
        x=0.5
    ),
    height=450,
    paper_bgcolor='white',
    plot_bgcolor='white',
    showlegend=False
)

fig.update_yaxes(range=[0, 100], row=1, col=1)

fig.show()

# Tabela resumo
print(f"\n{'='*70}")
print(f"📊 RESUMO POR DEPENDÊNCIA ADMINISTRATIVA")
print(f"{'='*70}")
print(f"{'Dependência':<15} {'Alunos':>12} {'Alfabetizados':>15} {'Taxa':>10}")
print(f"{'-'*55}")
for _, row in taxa_dep.iterrows():
    print(f"{row['desc_dependencia']:<15} {row['total_alunos']:>12,} {row['alfabetizados']:>15,} {row['taxa_alfabetizacao']:>9.1f}%")

### 7.4 📈 Análise da Proficiência em Língua Portuguesa

In [ ]:
# =============================================================================
# DISTRIBUIÇÃO DA PROFICIÊNCIA - ANÁLISE DETALHADA
# =============================================================================

PONTO_CORTE = 743  # Ponto de corte para alfabetização

# Separar grupos
prof_alf = df[df['ind_alfabetizado'] == 1]['proficiencia_lp'].dropna()
prof_nao = df[df['ind_alfabetizado'] == 0]['proficiencia_lp'].dropna()

fig = go.Figure()

# Histograma - Não Alfabetizados
fig.add_trace(go.Histogram(
    x=prof_nao,
    name='Não Alfabetizado',
    marker_color=cor_nao_alfabetizado,
    opacity=0.7,
    nbinsx=50
))

# Histograma - Alfabetizados
fig.add_trace(go.Histogram(
    x=prof_alf,
    name='Alfabetizado',
    marker_color=cor_alfabetizado,
    opacity=0.7,
    nbinsx=50
))

# Linha do ponto de corte
fig.add_vline(
    x=PONTO_CORTE, 
    line_dash="dash", 
    line_color="#F39237",
    line_width=3,
    annotation_text=f"Ponto de Corte: {PONTO_CORTE}",
    annotation_position="top",
    annotation_font_size=12
)

fig.update_layout(
    title=dict(
        text='<b>📈 Distribuição da Proficiência em Língua Portuguesa</b>',
        font=dict(size=18),
        x=0.5
    ),
    xaxis_title='Proficiência em LP',
    yaxis_title='Frequência',
    barmode='overlay',
    height=450,
    paper_bgcolor='white',
    plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    xaxis=dict(gridcolor='#E5E5E5'),
    yaxis=dict(gridcolor='#E5E5E5')
)

fig.show()

# Estatísticas
print(f"\n{'='*60}")
print(f"📊 ESTATÍSTICAS DE PROFICIÊNCIA")
print(f"{'='*60}")
print(f"\n🔹 Alfabetizados:")
print(f"   Média: {prof_alf.mean():.1f} | Mediana: {prof_alf.median():.1f}")
print(f"   Mín: {prof_alf.min():.1f} | Máx: {prof_alf.max():.1f}")
print(f"\n🔹 Não Alfabetizados:")
print(f"   Média: {prof_nao.mean():.1f} | Mediana: {prof_nao.median():.1f}")
print(f"   Mín: {prof_nao.min():.1f} | Máx: {prof_nao.max():.1f}")
print(f"\n📏 Diferença média: {prof_alf.mean() - prof_nao.mean():.1f} pontos")

In [ ]:
# =============================================================================
# BOXPLOT PROFICIÊNCIA POR DEPENDÊNCIA
# =============================================================================

fig = go.Figure()

for i, dep in enumerate(df['desc_dependencia'].dropna().unique()):
    dados_dep = df[df['desc_dependencia'] == dep]['proficiencia_lp'].dropna()
    fig.add_trace(go.Box(
        y=dados_dep,
        name=dep,
        marker_color=cores_dep[i % len(cores_dep)],
        boxmean=True
    ))

fig.add_hline(
    y=PONTO_CORTE, 
    line_dash="dash", 
    line_color="#F39237",
    line_width=2,
    annotation_text=f"Ponto de Corte ({PONTO_CORTE})",
    annotation_position="right"
)

fig.update_layout(
    title=dict(
        text='<b>📦 Proficiência por Dependência Administrativa</b>',
        font=dict(size=18),
        x=0.5
    ),
    yaxis_title='Proficiência em LP',
    height=450,
    paper_bgcolor='white',
    plot_bgcolor='white',
    showlegend=False,
    yaxis=dict(gridcolor='#E5E5E5')
)

fig.show()

### 7.5 🏗️ Análise de Infraestrutura Escolar

In [ ]:
# =============================================================================
# ANÁLISE DE INFRAESTRUTURA ESCOLAR
# =============================================================================

if 'score_infraestrutura' in df.columns:
    # Taxa por score de infraestrutura
    taxa_infra = df.groupby('score_infraestrutura').agg(
        total=('ind_alfabetizado', 'count'),
        alfabetizados=('ind_alfabetizado', 'sum')
    ).reset_index()
    taxa_infra['taxa'] = (taxa_infra['alfabetizados'] / taxa_infra['total'] * 100).round(2)
    taxa_infra = taxa_infra[taxa_infra['total'] >= 100]  # Filtrar amostras pequenas
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "bar"}, {"type": "histogram"}]],
        subplot_titles=('<b>Taxa de Alfabetização por Score</b>', '<b>Distribuição do Score</b>')
    )
    
    # Taxa por score
    fig.add_trace(
        go.Bar(
            x=taxa_infra['score_infraestrutura'],
            y=taxa_infra['taxa'],
            marker_color='#2E86AB',
            text=taxa_infra['taxa'].apply(lambda x: f'{x:.0f}%'),
            textposition='outside',
            showlegend=False
        ),
        row=1, col=1
    )
    
    # Linha de tendência (com tratamento de erro)
    try:
        x_data = taxa_infra['score_infraestrutura'].values.astype(float)
        y_data = taxa_infra['taxa'].values.astype(float)
        mask = ~(np.isnan(x_data) | np.isnan(y_data))
        if mask.sum() >= 2:
            z = np.polyfit(x_data[mask], y_data[mask], 1)
            p = np.poly1d(z)
            fig.add_trace(
                go.Scatter(
                    x=x_data[mask],
                    y=p(x_data[mask]),
                    mode='lines',
                    line=dict(color='#E94F37', width=3, dash='dash'),
                    name='Tendência',
                    showlegend=False
                ),
                row=1, col=1
            )
    except Exception as e:
        print(f'Linha de tendência não calculada: {e}')
    
    # Distribuição do score
    fig.add_trace(
        go.Histogram(
            x=df['score_infraestrutura'].dropna(),
            marker_color='#A23B72',
            nbinsx=15,
            showlegend=False
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title=dict(
            text='<b>🏗️ Impacto da Infraestrutura Escolar na Alfabetização</b>',
            font=dict(size=18),
            x=0.5
        ),
        height=400,
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    fig.update_yaxes(title_text='Taxa (%)', row=1, col=1)
    fig.update_xaxes(title_text='Score de Infraestrutura', row=1, col=1)
    fig.update_yaxes(title_text='Frequência', row=1, col=2)
    fig.update_xaxes(title_text='Score de Infraestrutura', row=1, col=2)
    
    fig.show()
    
    # Correlação
    corr = df[['score_infraestrutura', 'ind_alfabetizado']].dropna().corr().iloc[0, 1]
    print(f"\n📈 Correlação Score Infraestrutura vs Alfabetização: {corr:.4f}")
else:
    print("⚠️ Score de infraestrutura não disponível")

### 7.6 💰 Análise Socioeconômica

In [ ]:
# =============================================================================
# ANÁLISE INSE (INDICADOR SOCIOECONÔMICO)
# =============================================================================

# Buscar coluna INSE
inse_col = None
for col in df.columns:
    if 'inse' in col.lower():
        inse_col = col
        break

if inse_col and df[inse_col].notna().sum() > 0:
    # Criar faixas de INSE
    df['inse_faixa'] = pd.cut(df[inse_col], bins=5, labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto'])
    
    taxa_inse = df.groupby('inse_faixa').agg(
        total=('ind_alfabetizado', 'count'),
        alfabetizados=('ind_alfabetizado', 'sum')
    ).reset_index()
    taxa_inse['taxa'] = (taxa_inse['alfabetizados'] / taxa_inse['total'] * 100).round(2)
    
    cores_inse = ['#E94F37', '#F39237', '#FFC107', '#28A745', '#2E86AB']
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=taxa_inse['inse_faixa'].astype(str),
        y=taxa_inse['taxa'],
        marker_color=cores_inse,
        text=taxa_inse['taxa'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        textfont=dict(size=13)
    ))
    
    fig.update_layout(
        title=dict(
            text='<b>💰 Taxa de Alfabetização por Nível Socioeconômico (INSE)</b>',
            font=dict(size=18),
            x=0.5
        ),
        xaxis_title='Nível Socioeconômico',
        yaxis_title='Taxa de Alfabetização (%)',
        height=400,
        paper_bgcolor='white',
        plot_bgcolor='white',
        yaxis=dict(gridcolor='#E5E5E5', range=[0, 100])
    )
    
    fig.show()
    
    # Correlação
    corr = df[[inse_col, 'ind_alfabetizado']].dropna().corr().iloc[0, 1]
    print(f"\n📈 Correlação INSE vs Alfabetização: {corr:.4f}")
    
    df.drop('inse_faixa', axis=1, inplace=True)
else:
    print("ℹ️ Análise INSE não disponível")
    print("   O arquivo TS_ALUNO.csv original não contém dados de INSE.")
    print("   Para incluir esta análise, é necessário baixar os dados de INSE do INEP")
    print("   e fazer merge com a base de alunos por id_escola.")

### 7.7 🔗 Matriz de Correlação

In [ ]:
# =============================================================================
# MATRIZ DE CORRELAÇÃO - TOP VARIÁVEIS
# =============================================================================

# Selecionar colunas numéricas relevantes
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()

# Excluir colunas de ID
colunas_excluir = [c for c in colunas_numericas if any(x in c.lower() for x in 
                  ['id_', 'cod_', 'co_', 'bloco', 'gabarito', 'resposta', 'caderno', 'municipio', 'escola', 'microrregiao', 'mesorregiao'])]
colunas_analise = [c for c in colunas_numericas if c not in colunas_excluir]

# Limitar a 12 colunas mais correlacionadas com target
if len(colunas_analise) > 12:
    corr_target = df[colunas_analise].corrwith(df['ind_alfabetizado']).abs().sort_values(ascending=False)
    colunas_analise = corr_target.head(12).index.tolist()

# Calcular matriz
corr_matrix = df[colunas_analise].corr()

# Heatmap com Plotly
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu_r',
    zmid=0,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text}',
    textfont=dict(size=10),
    hovertemplate='%{x} vs %{y}<br>Correlação: %{z:.3f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='<b>🔗 Matriz de Correlação - Variáveis mais Relevantes</b>',
        font=dict(size=18),
        x=0.5
    ),
    height=600,
    width=800,
    paper_bgcolor='white'
)

fig.show()

In [ ]:
# =============================================================================
# TOP CORRELAÇÕES COM A VARIÁVEL ALVO
# =============================================================================

corr_target = df[colunas_analise].corrwith(df['ind_alfabetizado']).drop('ind_alfabetizado', errors='ignore').sort_values()

cores_corr = ['#2E86AB' if x > 0 else '#E94F37' for x in corr_target.values]

fig = go.Figure()

fig.add_trace(go.Bar(
    y=corr_target.index,
    x=corr_target.values,
    orientation='h',
    marker_color=cores_corr,
    text=corr_target.apply(lambda x: f'{x:.3f}'),
    textposition='outside',
    textfont=dict(size=11)
))

fig.add_vline(x=0, line_color='black', line_width=1)

fig.update_layout(
    title=dict(
        text='<b>📊 Correlação das Variáveis com ind_alfabetizado</b>',
        font=dict(size=18),
        x=0.5
    ),
    xaxis_title='Correlação de Pearson',
    height=500,
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#E5E5E5')
)

fig.show()

# Resumo
print(f"\n{'='*60}")
print(f"📊 TOP CORRELAÇÕES COM ind_alfabetizado")
print(f"{'='*60}")
print(f"\n🔹 Correlações Positivas (favorecem alfabetização):")
for col, val in corr_target.tail(5).iloc[::-1].items():
    print(f"   {col}: {val:+.4f}")
print(f"\n🔹 Correlações Negativas (desfavorecem alfabetização):")
for col, val in corr_target.head(5).items():
    print(f"   {col}: {val:+.4f}")

### 7.8 📋 Resumo Executivo dos Insights

In [ ]:
# =============================================================================
# DASHBOARD RESUMO - INSIGHTS PRINCIPAIS
# =============================================================================

# Calcular métricas
taxa_geral = df['ind_alfabetizado'].mean() * 100
melhor_uf = taxa_uf.iloc[-1]
pior_uf = taxa_uf.iloc[0]
melhor_dep = taxa_dep.iloc[0]
pior_dep = taxa_dep.iloc[-1]
prof_media_alf = df[df['ind_alfabetizado']==1]['proficiencia_lp'].mean()
prof_media_nao = df[df['ind_alfabetizado']==0]['proficiencia_lp'].mean()

# Criar figura de resumo
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=('Taxa Geral', 'Total de Alunos', 'Amplitude UF',
                   'Melhor UF', 'Pior UF', 'Gap Proficiência')
)

# Indicadores
fig.add_trace(go.Indicator(
    mode="number",
    value=taxa_geral,
    number={'suffix': '%', 'font': {'size': 40, 'color': '#2E86AB'}}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=len(df),
    number={'font': {'size': 40, 'color': '#2E86AB'}}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=melhor_uf['taxa_alfabetizacao'] - pior_uf['taxa_alfabetizacao'],
    number={'suffix': ' p.p.', 'font': {'size': 40, 'color': '#E94F37'}}
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode="number",
    value=melhor_uf['taxa_alfabetizacao'],
    number={'suffix': f'%<br><span style="font-size:14px">({melhor_uf["sigla_uf"]})</span>', 'font': {'size': 35, 'color': '#28A745'}}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=pior_uf['taxa_alfabetizacao'],
    number={'suffix': f'%<br><span style="font-size:14px">({pior_uf["sigla_uf"]})</span>', 'font': {'size': 35, 'color': '#E94F37'}}
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=prof_media_alf - prof_media_nao,
    number={'suffix': ' pts', 'font': {'size': 35, 'color': '#F39237'}}
), row=2, col=3)

fig.update_layout(
    title=dict(
        text='<b>📊 DASHBOARD RESUMO - PRINCIPAIS INDICADORES</b>',
        font=dict(size=22),
        x=0.5
    ),
    height=450,
    paper_bgcolor='white'
)

fig.show()

In [ ]:
# =============================================================================
# RESUMO TEXTUAL DOS INSIGHTS
# =============================================================================

print("\n" + "="*70)
print("📊 RESUMO EXECUTIVO - PRINCIPAIS INSIGHTS")
print("="*70)

print(f"""
🎯 VARIÁVEL ALVO
   • Taxa geral de alfabetização: {taxa_geral:.2f}%
   • Classe majoritária: {'Alfabetizado' if taxa_geral > 50 else 'Não Alfabetizado'}
   • Total de alunos analisados: {len(df):,}

🗺️ DESIGUALDADE GEOGRÁFICA
   • Amplitude entre UFs: {melhor_uf['taxa_alfabetizacao'] - pior_uf['taxa_alfabetizacao']:.1f} pontos percentuais
   • UF com maior taxa: {melhor_uf['sigla_uf']} ({melhor_uf['taxa_alfabetizacao']:.1f}%)
   • UF com menor taxa: {pior_uf['sigla_uf']} ({pior_uf['taxa_alfabetizacao']:.1f}%)

🏫 DEPENDÊNCIA ADMINISTRATIVA
   • Maior taxa: {melhor_dep['desc_dependencia']} ({melhor_dep['taxa_alfabetizacao']:.1f}%)
   • Menor taxa: {pior_dep['desc_dependencia']} ({pior_dep['taxa_alfabetizacao']:.1f}%)

📈 PROFICIÊNCIA EM LP
   • Média alfabetizados: {prof_media_alf:.1f} pontos
   • Média não alfabetizados: {prof_media_nao:.1f} pontos
   • Diferença: {prof_media_alf - prof_media_nao:.1f} pontos

💡 PRINCIPAIS CONCLUSÕES
   1. Existe forte desigualdade regional na alfabetização
   2. Escolas federais/privadas têm melhor desempenho
   3. Infraestrutura e nível socioeconômico impactam resultados
   4. Proficiência em LP é o principal preditor da alfabetização
""")

---
## 8. Próximos Passos

### ✅ O que foi feito:
1. **Bronze**: Carregamento de 6 bases de dados
2. **Silver**: Padronização, limpeza e remoção de colunas com anos antigos
3. **Gold**: Enriquecimento com 5 merges + feature engineering
4. **EDA**: Análise completa com 15+ visualizações profissionais

### 🚀 Próximos passos para Modelagem:
1. **Tratamento de Missing Values**: Imputação ou remoção estratégica
2. **Seleção de Features**: RFE, correlação, importância
3. **Balanceamento**: SMOTE, undersampling ou class weights
4. **Modelagem**: Random Forest, XGBoost, LightGBM
5. **Avaliação**: F1-Score, AUC-ROC, Precision, Recall
6. **Interpretabilidade**: SHAP values

In [ ]:
print("\n" + "="*70)
print("✅ NOTEBOOK EXECUTADO COM SUCESSO!")
print("="*70)
print(f"\n📁 Base exportada: {arquivo_saida}")
print(f"📊 Total de registros: {len(gold_alunos):,}")
print(f"📋 Total de variáveis: {len(gold_alunos.columns)}")
print(f"\n🎯 Pronto para modelagem preditiva!")